# Task 3: Customer Profile Building

Activity Level metrics:

Accounts are classified based on the total number of transactions recorded in the dataset.

- High Activity - 5 or more transactions 
- Medium Activity - 3 to 4 transactions    
- Low Activity - 1 to 2 transactions    

These metrics are defined for the purpose of customer segmentation and behavioral analysis.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the cleaned dataset from Task 1
df = pd.read_csv("goldman_sachs_cleaned.csv")

# Convert TransactionDate to datetime
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

# Create date columns (these will be useful later)
df["Year"] = df["TransactionDate"].dt.year
df["Month"] = df["TransactionDate"].dt.month
df["YearMonth"] = df["TransactionDate"].dt.to_period("M")

# Check the data
df.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,IsOverdraft,Year,Month,YearMonth
0,33,Cust6549,Acc12334,Credit,Withdrawal,Savings Account,Firm C,Central,Manager 1,2023-10-21,87480.05448,74008.43310,0.729101,319,200,False,2023,10,2023-10
1,177,Cust2942,Acc52650,Credit,Withdrawal,Home Loan,Firm A,East,Manager 3,2023-06-20,20315.74505,22715.83590,0.472424,692,47,False,2023,6,2023-06
2,178,Cust6776,Acc45101,Current,Deposit,Personal Loan,Firm C,South,Manager 3,2023-01-02,10484.57165,42706.09210,0.648784,543,109,False,2023,1,2023-01
3,173,Cust2539,Acc88252,Current,Withdrawal,Mutual Fund,Firm A,Central,Manager 2,2023-07-25,45122.27373,114176.56870,0.734832,430,103,False,2023,7,2023-07
4,67,Cust2626,Acc21878,Savings,Withdrawal,Home Loan,Firm C,Central,Manager 4,2023-07-25,42360.79878,17863.02644,0.289304,468,234,False,2023,7,2023-07


In [3]:
print(df.columns.tolist())

['TransactionID', 'CustomerID', 'AccountID', 'AccountType', 'TransactionType', 'Product', 'Firm', 'Region', 'Manager', 'TransactionDate', 'TransactionAmount', 'AccountBalance', 'RiskScore', 'CreditRating', 'TenureMonths', 'IsOverdraft', 'Year', 'Month', 'YearMonth']


In [4]:
#Counting transactions per account

txn_frequency = (
    df.groupby("AccountID")["TransactionID"]
      .count()
      .reset_index()
)

txn_frequency.columns = [
    "AccountID",
    "TransactionCount"
]

txn_frequency.head()

,AccountID,TransactionCount
0,Acc10117,4
1,Acc10996,5
2,Acc11062,2
3,Acc11188,5
4,Acc11285,3


In [5]:
# Assigning Activity Levels

def activity_level(count):
    
    if count >= 5:
        return "High"
    
    elif count >= 3:
        return "Medium"
    
    else:
        return "Low"


txn_frequency["ActivityLevel"] = (
    txn_frequency["TransactionCount"]
    .apply(activity_level)
)

txn_frequency.head()

,AccountID,TransactionCount,ActivityLevel
0,Acc10117,4,Medium
1,Acc10996,5,High
2,Acc11062,2,Low
3,Acc11188,5,High
4,Acc11285,3,Medium


In [6]:
# Checking the distribution 

txn_frequency["ActivityLevel"].value_counts()

ActivityLevel
Medium    76
High      73
Low       45
Name: count, dtype: int64

In [7]:
#Calculating Customer Statistics

account_stats = (
    df.groupby("AccountID")
      .agg(
          AvgBalance=("AccountBalance", "mean"),
          TotalVolume=("TransactionAmount", "sum"),
          TransactionCount=("TransactionID", "count")
      )
      .reset_index()
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume,TransactionCount
0,Acc10117,70107.007957,199480.967430,4
1,Acc10996,43568.008084,250739.550950,5
2,Acc11062,38137.132610,27189.136160,2
3,Acc11188,69652.151044,257576.603590,5
4,Acc11285,97401.348560,96729.609841,3


In [8]:
# Calculating the Median

balance_median = account_stats["AvgBalance"].median()

volume_median = account_stats["TotalVolume"].median()

print("Median Balance :", balance_median)

print("Median Transaction Volume :", volume_median)

Median Balance : 72044.73077833332
Median Transaction Volume : 201951.8756075


In [9]:
# Creating balance segments

account_stats["BalanceSegment"] = np.where(
    account_stats["AvgBalance"] >= balance_median,
    "High Balance",
    "Low Balance"
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment
0,Acc10117,70107.007957,199480.967430,4,Low Balance
1,Acc10996,43568.008084,250739.550950,5,Low Balance
2,Acc11062,38137.132610,27189.136160,2,Low Balance
3,Acc11188,69652.151044,257576.603590,5,Low Balance
4,Acc11285,97401.348560,96729.609841,3,High Balance


In [10]:
# Creating volume segments

account_stats["VolumeSegment"] = np.where(
    account_stats["TotalVolume"] >= volume_median,
    "High Volume",
    "Low Volume"
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment
0,Acc10117,70107.007957,199480.967430,4,Low Balance,Low Volume
1,Acc10996,43568.008084,250739.550950,5,Low Balance,High Volume
2,Acc11062,38137.132610,27189.136160,2,Low Balance,Low Volume
3,Acc11188,69652.151044,257576.603590,5,Low Balance,High Volume
4,Acc11285,97401.348560,96729.609841,3,High Balance,Low Volume


In [11]:
# Verifying the segments

print("Balance Segment Distribution")

print(account_stats["BalanceSegment"].value_counts())

print("\nVolume Segment Distribution")

print(account_stats["VolumeSegment"].value_counts())

Balance Segment Distribution
BalanceSegment
Low Balance     97
High Balance    97
Name: count, dtype: int64

Volume Segment Distribution
VolumeSegment
Low Volume     97
High Volume    97
Name: count, dtype: int64


In [12]:
# Displaying the final table

account_stats.head()


,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment
0,Acc10117,70107.007957,199480.967430,4,Low Balance,Low Volume
1,Acc10996,43568.008084,250739.550950,5,Low Balance,High Volume
2,Acc11062,38137.132610,27189.136160,2,Low Balance,Low Volume
3,Acc11188,69652.151044,257576.603590,5,Low Balance,High Volume
4,Acc11285,97401.348560,96729.609841,3,High Balance,Low Volume


# Customer Segmentation by Balance and Transaction Volume

Customers were segmented using the median values of Average Account Balance and Total Transaction Volume.

- Accounts with an average balance greater than or equal to the median were classified as High Balance; 
the remaining accounts were classified as Low Balance.
- Similarly, accounts with a total transaction volume greater than or equal to the median were classified as High Volume; 
the remaining accounts were classified as Low Volume.

Using the median provides a balanced and robust method for dividing customers into meaningful groups while minimizing the influence of 
extreme values.

In [14]:
# Merging Activity Level into account_stats

account_stats = account_stats.merge(
    txn_frequency[["AccountID", "ActivityLevel"]],
    on="AccountID",
    how="left"
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment,ActivityLevel
0,Acc10117,70107.007957,199480.967430,4,Low Balance,Low Volume,Medium
1,Acc10996,43568.008084,250739.550950,5,Low Balance,High Volume,High
2,Acc11062,38137.132610,27189.136160,2,Low Balance,Low Volume,Low
3,Acc11188,69652.151044,257576.603590,5,Low Balance,High Volume,High
4,Acc11285,97401.348560,96729.609841,3,High Balance,Low Volume,Medium


In [15]:
# Create FlowType

credit_types = ["Deposit"]

debit_types = ["Withdrawal", "Payment", "Transfer"]

df["FlowType"] = np.where(
    df["TransactionType"].isin(credit_types),
    "Credit",
    "Debit"
)

In [16]:
# Account summary

account_summary = (
    df.groupby(["AccountID", "FlowType"])["TransactionAmount"]
      .sum()
      .unstack(fill_value=0)
)

# Ensure both columns exist
account_summary = account_summary.reindex(
    columns=["Credit", "Debit"],
    fill_value=0
)

# Calculate Net Inflow
account_summary["Net Inflow"] = (
    account_summary["Credit"] -
    account_summary["Debit"]
)

account_summary.head()

FlowType,Credit,Debit,Net Inflow
AccountID,,,
Acc10117,142170.20378,57310.763650,84859.440130
Acc10996,62580.86356,188158.687390,-125577.823830
Acc11062,0.00000,27189.136160,-27189.136160
Acc11188,45748.34156,211828.262030,-166079.920470
Acc11285,0.00000,96729.609841,-96729.609841


In [17]:
account_stats = account_stats.merge(
    account_summary[["Net Inflow"]].reset_index(),
    on="AccountID",
    how="left"
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment,ActivityLevel,Net Inflow
0,Acc10117,70107.007957,199480.967430,4,Low Balance,Low Volume,Medium,84859.440130
1,Acc10996,43568.008084,250739.550950,5,Low Balance,High Volume,High,-125577.823830
2,Acc11062,38137.132610,27189.136160,2,Low Balance,Low Volume,Low,-27189.136160
3,Acc11188,69652.151044,257576.603590,5,Low Balance,High Volume,High,-166079.920470
4,Acc11285,97401.348560,96729.609841,3,High Balance,Low Volume,Medium,-96729.609841


In [ ]:
# Profile 1: High-Net Inflow Accounts

high_net_inflow = account_stats[
    account_stats["Net Inflow"] >
    account_stats["Net Inflow"].quantile(0.75)
]

print("Number of High-Net Inflow Accounts:",
      len(high_net_inflow))

high_net_inflow.head()

Number of High-Net Inflow Accounts: 49


,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment,ActivityLevel,Net Inflow
0,Acc10117,70107.007957,199480.96743,4,Low Balance,Low Volume,Medium,84859.44013
7,Acc12334,78082.517883,310227.28969,6,High Balance,High Volume,High,-11993.69447
10,Acc15359,118477.129717,243269.68088,3,High Balance,High Volume,Medium,2443.28512
17,Acc18140,39960.076965,84223.65859,2,Low Balance,Low Volume,Low,7138.16325
20,Acc19178,-1541.176812,64100.78213,1,Low Balance,Low Volume,Low,64100.78213


In [19]:
# Profile 2: High-Frequency, Low-Balance Accounts

high_freq_low_balance = account_stats[
    (account_stats["ActivityLevel"] == "High") &
    (account_stats["BalanceSegment"] == "Low Balance")
]

print("Number of High-Frequency, Low-Balance Accounts:",
      len(high_freq_low_balance))

high_freq_low_balance.head()

Number of High-Frequency, Low-Balance Accounts: 34


,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment,ActivityLevel,Net Inflow
1,Acc10996,43568.008084,250739.55095,5,Low Balance,High Volume,High,-125577.82383
3,Acc11188,69652.151044,257576.60359,5,Low Balance,High Volume,High,-166079.92047
8,Acc13357,69179.806513,432527.80826,6,Low Balance,High Volume,High,-328773.07592
18,Acc18177,63505.219474,249580.48543,5,Low Balance,High Volume,High,-27570.77895
29,Acc23736,60801.533102,309797.64287,7,Low Balance,High Volume,High,-120215.32337


In [ ]:
# Profile 3: Negative or Near-Zero Balance Accounts

''' 
Assumption
For this analysis, accounts with an average balance less than or equal to ₹1,000 are considered Near-Zero Balance Accounts.
This threshold has been defined for analytical purposes.

'''

near_zero_threshold = 1000

negative_or_near_zero = account_stats[
    account_stats["AvgBalance"] <= near_zero_threshold
]

print("Number of Negative/Near-Zero Balance Accounts:",
      len(negative_or_near_zero))

negative_or_near_zero.head()

Number of Negative/Near-Zero Balance Accounts: 1


,AccountID,AvgBalance,TotalVolume,TransactionCount,BalanceSegment,VolumeSegment,ActivityLevel,Net Inflow
20,Acc19178,-1541.176812,64100.78213,1,Low Balance,Low Volume,Low,64100.78213


In [21]:
# Display profile sizes

print("Customer Profile Summary")
print("-" * 35)

print("High-Net Inflow Accounts:",
      len(high_net_inflow))

print("High-Frequency Low-Balance Accounts:",
      len(high_freq_low_balance))

print("Negative/Near-Zero Balance Accounts:",
      len(negative_or_near_zero))

Customer Profile Summary
-----------------------------------
High-Net Inflow Accounts: 49
High-Frequency Low-Balance Accounts: 34
Negative/Near-Zero Balance Accounts: 1


# Customer Profiles

Three customer profiles were created based on transaction behaviour and account characteristics:

1. High-Net Inflow Accounts – Accounts belonging to the top 25% of net inflow values (Total 49 accounts).
2. High-Frequency, Low-Balance Accounts – Customers with high transaction frequency but low average account balances (Total 34 accounts).
3. Negative or Near-Zero Balance Accounts – Customers with an average account balance less than or equal to ₹1,000 (Total 1 account).

These profiles help identify valuable customers, highly active customers, and customers who may require financial monitoring or
targeted engagement strategies.
